# Batch .fcs File Analysis

Analyze multiple .fcs files together: consistent gating, cross-sample clustering, and comparative visualization.

## 1. Setup

In [ ]:
library(flowCore)
library(flowWorkspace)
library(ggcyto)
library(FlowSOM)
library(openCyto)
library(ggplot2)
library(dplyr)
library(tidyr)
library(uwot)
library(viridis)
library(patchwork)
library(pheatmap)

## 2. Load multiple .fcs files

In [ ]:
# -- Point this to a directory containing .fcs files --
fcs_dir <- "data/batch/"

fcs_files <- list.files(fcs_dir, pattern = "\\.fcs$", full.names = TRUE)
cat("Found", length(fcs_files), "FCS files:\n")
cat(paste(" ", basename(fcs_files), collapse = "\n"), "\n")

fs <- read.flowSet(fcs_files, transformation = FALSE, truncate_max_range = FALSE)
fs

## 3. Sample metadata

In [ ]:
# Create sample annotation
# -- Customize this for your experiment --
sample_info <- data.frame(
    name = sampleNames(fs),
    file = basename(fcs_files),
    # Add your own columns:
    # condition = c("ctrl", "treated", ...),
    # timepoint = c("0h", "24h", ...),
    stringsAsFactors = FALSE
)

sample_info

# Attach to flowSet
pData(fs)$name <- sample_info$name

## 4. Compensation & transformation (batch)

In [ ]:
# Identify fluorescence channels
all_channels <- colnames(fs)
scatter_pattern <- "^(FSC|SSC|Time)"
fluor_channels <- grep(scatter_pattern, all_channels, value = TRUE, invert = TRUE)

cat("Fluorescence channels:", paste(fluor_channels, collapse = ", "), "\n")

# Compensate each sample (uses embedded spillover if available)
fs_comp <- fsApply(fs, function(ff) {
    spill <- keyword(ff)$SPILL
    if (is.null(spill)) spill <- keyword(ff)$`$SPILLOVER`
    if (!is.null(spill)) compensate(ff, spill) else ff
})

# Transform (logicle)
lgcl <- estimateLogicle(fs_comp[[1]], channels = fluor_channels)
fs_trans <- transform(fs_comp, lgcl)

## 5. Build GatingSet & apply consistent gates

In [ ]:
gs <- GatingSet(fs_trans)

# Scatter gate — same thresholds across all samples
scatter_gate <- rectangleGate(
    filterId = "nonDebris",
    "FSC-A" = c(50000, 250000),
    "SSC-A" = c(5000, 200000)
)

gs_pop_add(gs, scatter_gate, parent = "root")
recompute(gs)

gs_pop_get_stats(gs)

## 6. Visualize gates across samples

In [ ]:
ggcyto(gs, aes(x = `FSC-A`, y = `SSC-A`)) +
    geom_hex(bins = 100) +
    geom_gate("nonDebris") +
    geom_stats() +
    scale_fill_viridis() +
    facet_wrap(~name) +
    theme_minimal()

## 7. Cross-sample population statistics

In [ ]:
pop_stats <- gs_pop_get_stats(gs, type = c("count", "freq"))
pop_stats

# Pivot for easy comparison
stats_wide <- pop_stats %>%
    select(sample, pop, freq = percent) %>%
    pivot_wider(names_from = pop, values_from = freq)

stats_wide

## 8. Merge & cluster with FlowSOM

In [ ]:
# Extract gated data from all samples, tag with sample ID
merged_list <- lapply(seq_along(sampleNames(gs)), function(i) {
    d <- exprs(gs_pop_get_data(gs, "nonDebris")[[i]])
    d <- as.data.frame(d)
    d$sample_id <- sampleNames(gs)[i]
    d
})

merged_df <- bind_rows(merged_list)
cat("Total gated events:", nrow(merged_df), "\n")

# Subsample for performance if needed
set.seed(42)
n_sub <- min(nrow(merged_df), 50000)
idx <- sample(nrow(merged_df), n_sub)
merged_sub <- merged_df[idx, ]

In [ ]:
# FlowSOM on merged data
mat <- as.matrix(merged_sub[, fluor_channels])

fsom_input <- flowFrame(mat)
fsom <- FlowSOM(fsom_input, colsToUse = fluor_channels, nClus = 12,
                xdim = 10, ydim = 10, seed = 42)

merged_sub$cluster <- factor(GetMetaclusters(fsom))
cat("Cluster sizes:\n")
table(merged_sub$cluster)

## 9. UMAP — colored by sample & cluster

In [ ]:
mat_scaled <- scale(mat)
set.seed(42)
umap_res <- uwot::umap(mat_scaled, n_neighbors = 15, min_dist = 0.2)

merged_sub$UMAP1 <- umap_res[, 1]
merged_sub$UMAP2 <- umap_res[, 2]

In [ ]:
p1 <- ggplot(merged_sub, aes(x = UMAP1, y = UMAP2, color = sample_id)) +
    geom_point(alpha = 0.2, size = 0.3) +
    theme_minimal() +
    labs(title = "UMAP — by sample")

p2 <- ggplot(merged_sub, aes(x = UMAP1, y = UMAP2, color = cluster)) +
    geom_point(alpha = 0.2, size = 0.3) +
    theme_minimal() +
    labs(title = "UMAP — FlowSOM clusters")

p1 / p2

## 10. Cluster abundance across samples

In [ ]:
abundance <- merged_sub %>%
    group_by(sample_id, cluster) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(sample_id) %>%
    mutate(freq = n / sum(n)) %>%
    ungroup()

ggplot(abundance, aes(x = cluster, y = freq, fill = sample_id)) +
    geom_bar(stat = "identity", position = "dodge") +
    theme_minimal() +
    labs(title = "Cluster frequency by sample", y = "Fraction")

## 11. Heatmap — median marker expression per cluster

In [ ]:
mfi <- merged_sub %>%
    group_by(cluster) %>%
    summarise(across(all_of(fluor_channels), median), .groups = "drop") %>%
    tibble::column_to_rownames("cluster") %>%
    as.matrix()

mfi_scaled <- t(scale(t(mfi)))

pheatmap(mfi_scaled,
         cluster_rows = TRUE,
         cluster_cols = TRUE,
         color = viridis(100),
         main = "Median marker expression per cluster (z-scored)")